In [1]:
import numpy as np  

In [2]:
%pip install -Uq pymcel

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pymcel as pc 

c:\Users\HP\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\HP\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Bienvenido a PyMCel v0.9.10 ¡al infinito y más allá!


In [4]:
%pip install -Uq rebound

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -Uq pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -Uq plotly

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install -Uq nbformat

Note: you may need to restart the kernel to use updated packages.


In [8]:
tabla_sol, jd_sol, X_sol = pc.consulta_horizons(
    id='Sun',
    location ='@SSB',
    epochs = '2026-02-26 00:00:00',
)

X_sol

array([-3.99267884e+08, -8.24090450e+08,  1.85429947e+07,  1.22108229e+01,
        1.23932702e+00, -2.42604119e-01])

In [9]:
tabla_jup, jd_jup, X_jup = pc.consulta_horizons(
    id='Jupiter Barycenter',
    location='@SSB',
    epochs='2026-02-26 00:00:00',
)

print("Jupiter position (X_jup):")
print(X_jup)

Jupiter position (X_jup):
[-3.13499559e+11  7.16531655e+11  4.04379637e+09 -1.21253772e+04
 -4.61863891e+03  2.90510926e+02]


In [10]:
r_sol_0 = X_sol[:3]  # Posición del Sol
r_jup_0 = X_jup[:3]  # Posición de Júpiter

v_sol_0 = X_sol[3:]  # Velocidad del Sol
v_jup_0 = X_jup[3:]  # Velocidad de Júpiter

In [11]:
deltat = 1*86400  # Un día en segundos

r_jup_dt = r_jup_0 + v_jup_0 * deltat
r_sol_dt = r_sol_0 + v_sol_0 * deltat

r_jup_0, r_jup_dt


(array([-3.13499559e+11,  7.16531655e+11,  4.04379637e+09]),
 array([-3.14547192e+11,  7.16132604e+11,  4.06889651e+09]))

In [12]:
import numpy as np 

In [13]:
mu_sol = pc.constantes.mu_sun
mu_jup = pc.constantes.mu_jupiter

rij_vec = r_jup_0 - r_sol_0
v_jup_dt = v_jup_0 - mu_sol * rij_vec / np.linalg.norm(rij_vec)**3 * deltat
v_jup_dt

array([-12117.89040582,  -4635.79206931,    290.41467508])

In [14]:
#Velocidad del Sol en t+deltat usando la misma aproximación

rij_vec_sol = r_sol_0 - r_jup_0
v_sol_dt = v_sol_0 - mu_jup * rij_vec_sol / np.linalg.norm(rij_vec_sol)**3 * deltat
v_sol_dt

array([12.2036746 ,  1.25570472, -0.24251222])

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

import matplotlib.pyplot as plt

# Initialize arrays to store positions
years = 12
days_per_year = 365.25
total_days = int(years * days_per_year)

# Store positions over time
jup_positions = np.zeros((total_days, 3))
sol_positions = np.zeros((total_days, 3))

# Store current state
r_jup = r_jup_0.copy()
v_jup = v_jup_0.copy()
r_sol = r_sol_0.copy()
v_sol = v_sol_0.copy()

# Iterative integration
for i in range(total_days):
    jup_positions[i] = r_jup
    sol_positions[i] = r_sol
    
    # Update positions
    r_jup = r_jup + v_jup * deltat
    r_sol = r_sol + v_sol * deltat
    
    # Update velocities using gravitational interactions
    rij_vec = r_jup - r_sol
    v_jup = v_jup - mu_sol * rij_vec / np.linalg.norm(rij_vec)**3 * deltat
    
    rij_vec_sol = r_sol - r_jup
    v_sol = v_sol - mu_jup * rij_vec_sol / np.linalg.norm(rij_vec_sol)**3 * deltat

# Create 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot orbits
ax.plot(jup_positions[:, 0], jup_positions[:, 1], jup_positions[:, 2], 'b-', label='Jupiter', linewidth=1)
ax.plot(sol_positions[:, 0], sol_positions[:, 1], sol_positions[:, 2], 'y-', label='Sun', linewidth=1)

# Plot starting points
ax.scatter(*r_jup_0, color='blue', s=100, marker='o', label='Jupiter Start')
ax.scatter(*r_sol_0, color='orange', s=100, marker='o', label='Sun Start')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title(f'Jupiter and Sun Orbits ({years} years)')
ax.legend()
plt.tight_layout()
plt.show()